<a href="https://colab.research.google.com/github/samcamacho62-dotcom/-ITAI-1371-ML-Labs/blob/main/lab03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Load the dataset and create the DataFrame (df)
wine_data = load_wine()
df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
df['wine_class'] = wine_data.target

# Recreate the baseline model from Part 3 & 4 so we have something to compare against
X_baseline = df[['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash']]
y = df['wine_class']
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42, stratify=y
)
baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(X_train_base, y_train_base)
baseline_accuracy = accuracy_score(y_test_base, baseline_model.predict(X_test_base))
results = {'Logistic Regression': baseline_accuracy}

print("✅ Setup complete! You can now run Task 1.")

✅ Setup complete! You can now run Task 1.


In [3]:
# Task 1: Try different features
print("Task 1: Experiment with Different Features")
print("=" * 40)

# Your chosen features
your_features = ['alcohol', 'color_intensity', 'proline']

# Build model with your features
X_your = df[your_features]
y = df['wine_class']

X_train_your, X_test_your, y_train_your, y_test_your = train_test_split(
    X_your, y, test_size=0.2, random_state=42, stratify=y
)

# Train a logistic regression model
your_model = LogisticRegression(random_state=42)
your_model.fit(X_train_your, y_train_your)

# Evaluate
y_pred_your = your_model.predict(X_test_your)
your_accuracy = accuracy_score(y_test_your, y_pred_your)

print(f"\nYour model features: {your_features}")
print(f"Your model accuracy: {your_accuracy:.3f} ({your_accuracy*100:.1f}%)")

# Compare with original model
print(f"Original model accuracy: {results['Logistic Regression']:.3f}")

if your_accuracy > results['Logistic Regression']:
    print("🎉 Great job! Your feature selection improved the model!")
else:
    print("🤔 Try different features to see if you can improve performance!")

Task 1: Experiment with Different Features

Your model features: ['alcohol', 'color_intensity', 'proline']
Your model accuracy: 0.833 (83.3%)
Original model accuracy: 0.889
🤔 Try different features to see if you can improve performance!


In [4]:
from sklearn.preprocessing import StandardScaler

# Combine the features from the baseline and your custom selection
combined_features = [
    'alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash',
    'color_intensity', 'proline'
]

X_combined = df[combined_features]
y = df['wine_class']

# Split the data
X_train_comb, X_test_comb, y_train_comb, y_test_comb = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the training data, then transform both train and test sets
X_train_scaled = scaler.fit_transform(X_train_comb)
X_test_scaled = scaler.transform(X_test_comb)

# Train the logistic regression model on the scaled data
optimized_model = LogisticRegression(random_state=42)
optimized_model.fit(X_train_scaled, y_train_comb)

# Evaluate the new model
y_pred_opt = optimized_model.predict(X_test_scaled)
opt_accuracy = accuracy_score(y_test_comb, y_pred_opt)

print("Task: Improve Performance with Scaling and Combined Features")
print("=" * 60)
print(f"Features used: {combined_features}")
print(f"Optimized model accuracy: {opt_accuracy:.3f} ({opt_accuracy*100:.1f}%)")
print(f"Original baseline accuracy: {results['Logistic Regression']:.3f} (88.9%)")

if opt_accuracy > results['Logistic Regression']:
    print("\n🚀 Incredible! Combining features and scaling the data successfully improved the model!")
else:
    print("\n🤔 The baseline is still holding strong.")

Task: Improve Performance with Scaling and Combined Features
Features used: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'color_intensity', 'proline']
Optimized model accuracy: 0.917 (91.7%)
Original baseline accuracy: 0.889 (88.9%)

🚀 Incredible! Combining features and scaling the data successfully improved the model!


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Train a Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train_comb)
rf_pred = rf_model.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test_comb, rf_pred)

# Train a Support Vector Machine (SVM) model
svm_model = SVC(random_state=42)
svm_model.fit(X_train_scaled, y_train_comb)
svm_pred = svm_model.predict(X_test_scaled)
svm_accuracy = accuracy_score(y_test_comb, svm_pred)

print("Task: Advanced Models (Random Forest & SVM)")
print("=" * 60)
print(f"Random Forest accuracy: {rf_accuracy:.3f} ({rf_accuracy*100:.1f}%)")
print(f"SVM accuracy:         {svm_accuracy:.3f} ({svm_accuracy*100:.1f}%)")
print(f"Previous best (LogReg): {opt_accuracy:.3f} ({opt_accuracy*100:.1f}%)")

if rf_accuracy > opt_accuracy or svm_accuracy > opt_accuracy:
    print("\n🚀 Success! We pushed the accuracy even higher.")
else:
    print("\n🤔 Logistic regression held its ground against the advanced models.")

Task: Advanced Models (Random Forest & SVM)
Random Forest accuracy: 0.917 (91.7%)
SVM accuracy:         0.944 (94.4%)
Previous best (LogReg): 0.917 (91.7%)

🚀 Success! We pushed the accuracy even higher.


In [6]:
from sklearn.model_selection import GridSearchCV

# Define the "grid" of settings we want to test
param_grid = {
    'C': [0.1, 1, 10, 100],          # Controls how strictly the model avoids errors
    'gamma': [1, 0.1, 0.01, 0.001],  # Controls the influence of individual data points
    'kernel': ['rbf', 'linear']      # The mathematical shape of the decision boundary
}

# Initialize the Grid Search with our SVM model
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, verbose=1, n_jobs=-1)

# Run the search (this trains dozens of models automatically!)
grid_search.fit(X_train_scaled, y_train_comb)

# Get the absolute best model from the search
best_svm = grid_search.best_estimator_

# Evaluate the optimized model
y_pred_best_svm = best_svm.predict(X_test_scaled)
best_svm_accuracy = accuracy_score(y_test_comb, y_pred_best_svm)

print("\nTask: Hyperparameter Tuning with GridSearchCV")
print("=" * 60)
print(f"Best settings found: {grid_search.best_params_}")
print(f"Tuned SVM accuracy:   {best_svm_accuracy:.3f} ({best_svm_accuracy*100:.1f}%)")
print(f"Previous best (SVM):  {svm_accuracy:.3f} (94.4%)")

if best_svm_accuracy > svm_accuracy:
    print("\n🏆 INCREDIBLE! You optimized the model and beat the high score!")
elif best_svm_accuracy == svm_accuracy:
    print("\n🎯 The model is completely maxed out! We found its peak performance.")
else:
    print("\n🤔 The default settings were surprisingly already perfect.")

Fitting 5 folds for each of 32 candidates, totalling 160 fits

Task: Hyperparameter Tuning with GridSearchCV
Best settings found: {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
Tuned SVM accuracy:   0.917 (91.7%)
Previous best (SVM):  0.944 (94.4%)

🤔 The default settings were surprisingly already perfect.


### Your Reflection and Analysis

**My Analysis of the Wine Classification Project**
*   **Best performing model:** The default Support Vector Machine (SVM) model using scaled data and a combined feature set, which achieved 94.4% accuracy.
*   **Why do you think this model performed better?:** SVM algorithms excel at finding complex, non-linear boundaries in high-dimensional data. Interestingly, the default SVM parameters generalized better to the unseen test data than the parameters chosen by GridSearchCV, which slightly overfitted the training data and caused a performance drop.
*   **What would you try next to improve performance?:** To capture that final 5.6%, I could acquire a larger dataset to prevent the hyperparameter tuning from overfitting, experiment with different scaling techniques (like `MinMaxScaler`), or utilize an ensemble method that combines the predictions of both the Logistic Regression and SVM models.